# Multimodal Evaluation: Images, Audio, and PDF

[Read the cookbook](https://docs.futureagi.com/docs/cookbook/quickstart/multimodal-eval). Create your own image, audio clip and packing slip, then compare their hosted scores. You need FFmpeg with its `flite` filter on the kernel host. Start in an empty directory so `returns.wav` does not already exist. Use a fresh Python 3.12 kernel. Hosted evaluations can consume credits.

## Install

In [ ]:
import subprocess
import sys

subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "agent-learning-kit==0.3.7", "Pillow==11.3.0"], check=True)

In [ ]:
from getpass import getpass
import os

for name in ('FI_API_KEY', 'FI_SECRET_KEY'):
    if not os.environ.get(name):
        os.environ[name] = getpass(f"{name}: ")
    if not os.environ[name].strip():
        raise ValueError(f"{name} is required")

## Set up the Evaluator

In [ ]:
import base64
import io
import json
import subprocess
from pathlib import Path

from PIL import Image, ImageDraw
from fi.evals import Evaluator

evaluator = Evaluator()
judgments = []

def media_uri(mime, content):
    return f"data:{mime};base64," + base64.b64encode(content).decode("ascii")

def score(template, inputs, model):
    batch = evaluator.evaluate(eval_templates=template, inputs=inputs, model_name=model)
    row = batch.eval_results[0]
    if row is None or row.output is None or not row.eval_id:
        raise RuntimeError(row.reason if row else "No judgment returned")
    judgments.append({"name": template, "eval_id": row.eval_id, "verdict": row.output})
    print(f"{template}: {row.output}\nReason: {row.reason}")

## Detect caption hallucination

In [ ]:
icon = Image.new("RGB", (192, 192), "white")
ImageDraw.Draw(icon).polygon([(96, 20), (26, 162), (166, 162)], fill="red")
buffer = io.BytesIO()
icon.save(buffer, format="PNG")
image = media_uri("image/png", buffer.getvalue())
score("caption_hallucination", {"image": image, "caption": "A red triangle on a white background."}, "turing_small")

Keep the image fixed and change only the caption.

In [ ]:
score("caption_hallucination", {"image": image, "caption": "A blue bicycle leaning against a tree."}, "turing_small")

## Detect synthetic images

In [ ]:
score("synthetic_image_evaluator", {"image": image}, "turing_small")

## Evaluate audio quality

In [ ]:
text = "Unused gear can be returned within thirty days."
subprocess.run([
    "ffmpeg", "-v", "error", "-f", "lavfi", "-i", f"flite=text={text}:voice=slt",
    "-ac", "1", "-ar", "16000", "returns.wav",
], check=True)
audio = media_uri("audio/wav", Path("returns.wav").read_bytes())
score("audio_quality", {"input_audio": audio}, "turing_large")

## Evaluate text-to-speech accuracy

In [ ]:
score("TTS_accuracy", {"text": text, "generated_audio": audio}, "turing_large")

## Evaluate OCR output against a PDF

In [ ]:
slip = Image.new("RGB", (850, 1100), "white")
ImageDraw.Draw(slip).multiline_text(
    (50, 60), "Ridgeline Outfitters\nPacking slip: RO-1042\nItem: Trail mug\nQuantity: 2",
    fill="black", font_size=36, spacing=12,
)
buffer = io.BytesIO()
slip.save(buffer, format="PDF", resolution=100)
score("ocr_evaluation", {
    "input_pdf": media_uri("application/pdf", buffer.getvalue()),
    "json_content": json.dumps({"packing_slip": "RO-1042", "item": "Trail mug", "quantity": 2}),
}, "turing_large")
Path("multimodal-results.json").write_text(json.dumps(judgments, indent=2), encoding="utf-8")

`multimodal-results.json` contains six evaluation IDs and verdicts. To score your own dataset in the dashboard, select **Add Evaluation** and map the required media and text columns.